In [1]:
!pip install pandas numpy sentence-transformers faiss-cpu


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("Sales Data.csv")

df = df.drop(columns=["Unnamed: 0"], errors="ignore")

df["Order Date"] = pd.to_datetime(df["Order Date"])

df["Month"] = df["Order Date"].dt.month
df["Hour"] = df["Order Date"].dt.hour
df["Year"] = df["Order Date"].dt.year

In [23]:
product_df = df.groupby("Product").agg({
    "Sales": "sum",
    "Price Each": "mean",
    "Quantity Ordered": "sum"
}).reset_index()

product_df.head()

,Product,Sales,Price Each,Quantity Ordered
0,20in Monitor,454148.71,109.99,4129
1,27in 4K Gaming Monitor,2435097.56,389.99,6244
2,27in FHD Monitor,1132424.50,149.99,7550
3,34in Ultrawide Monitor,2355558.01,379.99,6199
4,AA Batteries (4-pack),106118.40,3.84,27635


In [31]:
def create_product_text(row):
    return f"""
    Product: {row['Product']}.
    Category: electronic device or accessory.
    Related to smartphones, laptops, headphones, chargers, cables and digital devices.
    Total sales: {row['Sales']}.
    Average price: {row['Price Each']}.
    Total quantity sold: {row['Quantity Ordered']}.
    """.strip()

product_df["text"] = product_df.apply(create_product_text, axis=1)

In [37]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

vectors = model.encode(product_df["text"].tolist())

In [38]:
import faiss

vectors = np.array(vectors).astype("float32")

# normalize vectors
faiss.normalize_L2(vectors)

index = faiss.IndexFlatIP(vectors.shape[1])
index.add(vectors)

print("Vectors:", index.ntotal)

Vectors: 19


In [39]:
metadata = product_df.to_dict(orient="records")

print("Metadata:", len(metadata))

Metadata: 19


In [40]:
def search(query, top_k=5):
    query = f"{query} electronic device or accessory"

    query_vec = model.encode([query])
    query_vec = np.array(query_vec).astype("float32")

    faiss.normalize_L2(query_vec)

    distances, indices = index.search(query_vec, top_k)

    results = []
    for i in indices[0]:
        if i < len(metadata):   # prevents error
            results.append(metadata[i])

    return results

In [41]:
print(index.ntotal)
print(len(metadata))

19
19


In [42]:
results = search("phone", top_k=10)

for r in results:
    print(r["Product"])

iPhone
Google Phone
Vareebadd Phone
LG Dryer
AA Batteries (4-pack)
20in Monitor
Flatscreen TV
USB-C Charging Cable
Wired Headphones
27in 4K Gaming Monitor


In [43]:
print("\nLaptop:")
for r in search("laptop"):
    print(r["Product"])


Laptop:
ThinkPad Laptop
Macbook Pro Laptop
20in Monitor
27in 4K Gaming Monitor
iPhone
